# Deploy latest `main` → VM

**One-click notebook to pull the latest code from `main` and restart all `ict-*` systemd services.**

Use this whenever Claude tells you to run a `git pull` and restart services on the VM — for example after a bug fix is merged.

**How to run:** `Runtime → Run all`. Approve the Drive access dialog when it pops up (first time per session only). No other interaction needed.

**Required Colab Secrets** (set once via `Tools → Secrets`, toggle *Notebook access* on):

| Name | What it holds |
|---|---|
| `VM_SSH_HOST` | The VM's hostname or public IP |
| `VM_SSH_USER` | SSH user on the VM (usually `ubuntu`) |

**Required SSH key file** — place in Google Drive at:

`My Drive/ICT_Bot_Secrets/ict-bot-ovm-private.key`

If Drive isn't available, a file-picker will pop up so you can upload the key from your computer.

In [ ]:
# Step 1: Mount Google Drive (one-click Allow dialog on first run).
import os
from google.colab import drive

print('⏳ Mounting Google Drive…')
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'⚠️  drive.mount() raised: {exc}')

if not os.path.exists('/content/drive/MyDrive'):
    try:
        drive.mount('/content/drive', force_remount=True)
    except Exception as exc:
        print(f'⚠️  force-remount also failed: {exc}')

drive_mounted = os.path.exists('/content/drive/MyDrive')
print('✅ Drive mounted.' if drive_mounted else '⚠️  Drive NOT mounted — will fall back to file-picker.')

In [ ]:
# Step 2: Locate the SSH private key.
import os, shutil, stat, tempfile
from google.colab import userdata

DRIVE_FOLDER = '/content/drive/MyDrive/ICT_Bot_Secrets'
DEFAULT_NAMES = ['ict-bot-ovm-private.key', 'vm_ssh_key', 'id_rsa', 'id_ed25519', 'id_ecdsa']

try:
    override = userdata.get('SSH_KEY_FILE')
except Exception:
    override = None
candidates = ([override] if override else []) + DEFAULT_NAMES

ssh_key_path = None
if drive_mounted:
    for name in candidates:
        p = os.path.join(DRIVE_FOLDER, name)
        if os.path.exists(p):
            ssh_key_path = p
            break

if ssh_key_path is None:
    for name in candidates:
        p = os.path.join('/content', name)
        if os.path.exists(p):
            ssh_key_path = p
            break

if ssh_key_path is None:
    print('SSH key not found in Drive. Upload it now:')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file uploaded — re-run and pick the key file.')
    ssh_key_path = os.path.join('/content', next(iter(uploaded.keys())))

with open(ssh_key_path) as f:
    first = f.readline().strip()
if not first.startswith('-----BEGIN') or 'PUBLIC' in first:
    raise SystemExit(f'This looks like a public key or not a key at all: {first[:60]}')

print(f'✅ SSH key found: {ssh_key_path}')

In [ ]:
# Step 3: Load VM connection details from Colab Secrets.
from google.colab import userdata

VM_HOST = userdata.get('VM_SSH_HOST')
VM_USER = userdata.get('VM_SSH_USER')
REPO_DIR = f'/home/{VM_USER}/ict-trading-bot'

if not VM_HOST or not VM_USER:
    raise SystemExit('VM_SSH_HOST and VM_SSH_USER must be set in Colab Secrets (Tools → Secrets).')

print(f'✅ Target: {VM_USER}@{VM_HOST}  repo: {REPO_DIR}')

In [ ]:
# Step 4: git pull + restart services.
import os, shutil, stat, subprocess, tempfile

SERVICES = [
    'ict-trader-live.service',
    'ict-telegram-bot.service',
    'ict-claude-bridge.service',   # Claude comms bridge (BUG-058/059 fix lives here)
]

tmpdir = tempfile.mkdtemp(prefix='deploy-main-')
key_path = os.path.join(tmpdir, 'vm_key')
shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)

ssh_opts = [
    '-i', key_path,
    '-o', 'StrictHostKeyChecking=accept-new',
    '-o', f'UserKnownHostsFile={tmpdir}/known_hosts',
    '-o', 'ConnectTimeout=15',
    '-o', 'BatchMode=yes',
]
target = f'{VM_USER}@{VM_HOST}'

def ssh(cmd, label, allow_fail=False):
    proc = subprocess.run(['ssh'] + ssh_opts + [target, cmd], capture_output=True, text=True)
    out = (proc.stdout or '').strip()
    err = (proc.stderr or '').strip()
    if proc.returncode != 0 and not allow_fail:
        raise SystemExit(f'{label} failed (exit {proc.returncode}):\n{err or out}')
    return out

try:
    # Connectivity check
    print(f'⏳ Connecting to {target}…')
    ssh('echo connected', 'SSH connectivity')
    print('✅ SSH OK')

    # git pull
    print(f'⏳ git pull in {REPO_DIR}…')
    pull_out = ssh(
        f'cd {REPO_DIR} && git fetch --quiet origin main && git merge --ff-only origin/main 2>&1',
        'git pull', allow_fail=False
    )
    print(f'✅ git pull: {pull_out or "already up to date"}')

    # Restart each service
    for svc in SERVICES:
        print(f'⏳ Restarting {svc}…')
        ssh(f'sudo -n systemctl restart {svc}', f'restart {svc}', allow_fail=True)
        state = ssh(f'sudo -n systemctl is-active {svc}', f'is-active {svc}', allow_fail=True)
        if state == 'active':
            print(f'✅ {svc}: active')
        elif state in ('inactive', 'failed', ''):
            print(f'⚠️  {svc}: {state or "not installed"} — may not be configured on this VM yet, or check journalctl.')
        else:
            print(f'ℹ️  {svc}: {state}')

finally:
    shutil.rmtree(tmpdir, ignore_errors=True)

print()
print('Done. Latest main is deployed. Open Telegram and check:')
print('  /status  — trader should show normal pipeline results')
print('  /accounts_status  — each account should show a real balance')

## Verification

After the cells above complete, in Telegram:

1. **`/accounts_status`** — every account should show ✅ with a real USDT balance.
2. **Watch one `Pipeline result` tick** — every line should carry `strategy=…`.
3. **`/smoke_test`** — each account should return ✅ `rejected_too_small`.

If `ict-claude-bridge.service` shows `inactive` or `failed`, it may not yet be enabled on the VM. To enable it once:

```bash
sudo systemctl enable --now ict-claude-bridge.service
```

## When to re-run

- Claude tells you to deploy a fix (e.g. after a bug-fix PR merges to `main`).
- The bridge bot stops sending Telegram messages.
- You want to pick up any recent `main` changes without rotating API keys.